# ABC Credit: feature engineering, selection and model packaging

**Purpose:** go from the raw application data to a production-ready Approve/Decline pipeline that the chatbot can call in real time, using as **few customer questions** as possible.

**How this connects to the cleaning step:** the rules in `data_cleaning.ipynb` are packaged in [`credit_pipeline.py`](credit_pipeline.py), which is the `clean_transform` + params of the brief:
- `basic_clean`: fixed rules, no learning.
- `DomainImputer`: LTV and tier lookups, learned from **train only**.

Column names in this notebook vs the brief:

| Brief | Here |
|---|---|
| `Pincode_Region` | `pin_district` (first 3 digits) and `pin_zone` (first digit) |
| `Tier_Num` | `tier_rank` |
| `Decision` | `y` (1 = DECLINED, 0 = APPROVED) |

We start from the **raw** file, not a pre-cleaned CSV. That way the saved pipeline can take a raw chatbot record, and the cleaning is fitted after the split, so nothing leaks.

**Where each field comes from in the live journey:**

| Source | Fields | Asked to the customer? |
|---|---|---|
| Customer (chatbot) | Qualifications, Employment_Type, Age, Pincode, Gender, Resident_Type, Net_salary, PAST_LOANS_ACTIVE | **Yes**: these are the questions we try to minimise |
| Dealer quote | Loan_Amount, LTV (i.e. price and down payment) | No |
| Dealer catalogue | Product_Code, Make_Code, Model_Description, Model_Variant | No |
| Lookup | Final_Tier (from Pincode) | No |

All outputs are written to `pilot/artifacts/`.

In [1]:
# ---- Setup ----
import os, time, warnings, json
import numpy as np
import pandas as pd
import joblib
from IPython.display import display

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_predict
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, average_precision_score, recall_score, precision_score, fbeta_score
from sklearn.inspection import permutation_importance
from scipy.stats import chi2_contingency
from lightgbm import LGBMClassifier
from imblearn.pipeline import Pipeline as ImbPipeline
from imblearn.over_sampling import SMOTE
from imblearn.under_sampling import RandomUnderSampler

import credit_pipeline as cp
from credit_pipeline import (RawCleaner, DomainImputer, FeatureEngineer, basic_clean,
                             training_row_filter, build_preprocessor, processing_pipeline, cramers_v)

warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=FutureWarning)
pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 200)

RANDOM_STATE = 42                     # one seed everywhere
np.random.seed(RANDOM_STATE)
ART = "artifacts"
os.makedirs(ART, exist_ok=True)
CV = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

# Loan assumptions for EMI (judgment call, stated for the panel):
TENURE_MONTHS = 24     # typical two-wheeler tenure (common range 12-36)
ANNUAL_RATE = 0.22     # 22% p.a. reducing balance; NBFC two-wheeler rates are roughly 18-26%

## Step 1: Load and split first (prevent leakage)

1. Read the raw file as text.
2. Apply the **training-only** row filters (empty row, duplicates).
3. Split 80/20, stratified.

Nothing below this cell looks at the validation rows until evaluation.

In [2]:
raw = pd.read_csv("../data/training_data.csv", dtype=str, keep_default_na=False)
data = training_row_filter(raw)

y = (data["Decision"] == "DECLINED").astype(int).rename("y")    # 1 = DECLINED
X = data.drop(columns=["Decision", "ID"])                        # raw inputs exactly as received

X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.20, stratify=y, random_state=RANDOM_STATE)

counts = pd.DataFrame({
    "rows": [len(y_train), len(y_val)],
    "APPROVED (0)": [(y_train == 0).sum(), (y_val == 0).sum()],
    "DECLINED (1)": [(y_train == 1).sum(), (y_val == 1).sum()],
    "decline rate": [f"{y_train.mean():.2%}", f"{y_val.mean():.2%}"],
}, index=["train", "validation"])
print("X_train", X_train.shape, "| X_val", X_val.shape)
counts

rows: start 108,423 | empty -1 | no Decision -0 | duplicates -13 | kept 108,409
X_train (86727, 15) | X_val (21682, 15)


,rows,APPROVED (0),DECLINED (1),decline rate
train,86727,82169,4558,5.26%
validation,21682,20542,1140,5.26%


## Step 2: Feature engineering

Each feature describes **affordability, commitment or stability**, the classic credit-risk dimensions. They are built in `FeatureEngineer`, which has no learned parameters. Every ratio guards against a zero or missing salary (the result becomes *missing*, never infinite), and any ±∞ is replaced with missing.

| Feature | Business logic |
|---|---|
| `Loan_to_Income` = Loan ÷ monthly salary | How many months of salary are borrowed. Higher means less affordable. |
| `Vehicle_Price` = Loan ÷ (LTV/100) | Price of the bike. A pricier bike relative to profile means more stretch. |
| `Down_Payment` = Price − Loan | The customer's own money in the deal ("skin in the game"). People who put money down default less. |
| `Approx_EMI` | Monthly instalment at **24 months, 22% p.a.** (stated assumption): EMI = P·r(1+r)ⁿ / ((1+r)ⁿ−1) |
| `EMI_to_Income` | Share of salary going to this EMI. This is the FOIR idea lenders use. |
| `Residual_Income` = Salary − EMI | Money left to live on after the EMI. |
| `Down_Payment_to_Income` | Down payment in months of salary, a proxy for savings capacity. |
| `Age_Band` | 18-25 / 26-35 / 36-45 / 46-60 / 60+. A binned age that lets the linear model capture a non-straight-line effect. |
| `Is_First_Time_Borrower` | 1 if no past loans. A thin credit file means less evidence of repayment behaviour. |
| `High_LTV` | LTV ≥ 90%: almost no down payment, a common policy red flag. |
| `Is_Stable_Income`, `Owns_Home`, `Is_EV` | Salaried/pensioner; owns residence; electric vehicle. Stability and segment markers. |

The `DomainImputer` lookups are fitted on **train only**.

In [3]:
# Fit the learned cleaning step on TRAIN only, then apply to both parts
from sklearn.pipeline import Pipeline
clean_fe = Pipeline([("clean", RawCleaner()), ("domain", DomainImputer()),
                     ("features", FeatureEngineer(TENURE_MONTHS, ANNUAL_RATE))])
F_train = clean_fe.fit_transform(X_train)
F_val = clean_fe.transform(X_val)
print("Engineered train:", F_train.shape, "| validation:", F_val.shape)

NEW_FEATURES = ["Loan_to_Income", "Vehicle_Price", "Down_Payment", "Approx_EMI", "EMI_to_Income",
                "Residual_Income", "Down_Payment_to_Income", "Age_Band", "Is_First_Time_Borrower",
                "High_LTV", "Is_Stable_Income", "Owns_Home", "Is_EV"]
num = F_train[NEW_FEATURES].select_dtypes("number")
print("Infinite values left:", int(np.isinf(num.to_numpy(dtype=float)).sum()))
F_train[NEW_FEATURES].describe(include="all").T[["count", "mean", "min", "50%", "max", "unique"]]

Engineered train: (86727, 37) | validation: (21682, 37)
Infinite values left: 0


,count,mean,min,50%,max,unique
Loan_to_Income,86184.0,25.897476,0.084903,2.761427,131203.0,NaN
Vehicle_Price,86727.0,115584.732004,50510.982659,116575.88026,363111.517367,NaN
Down_Payment,86727.0,26294.788526,378.542169,21395.651608,263800.517367,NaN
Approx_EMI,86727.0,4632.197496,901.071668,4670.95341,14881.093026,NaN
EMI_to_Income,86184.0,1.343513,0.004405,0.143258,6806.569524,NaN
Residual_Income,86676.0,29349.13994,-13566.13744,26695.128097,975520.061955,NaN
Down_Payment_to_Income,86184.0,8.756148,0.012618,0.6709,73130.257481,NaN
Age_Band,86715,NaN,NaN,NaN,NaN,5
Is_First_Time_Borrower,86727.0,0.531945,0.0,1.0,1.0,NaN
High_LTV,86727.0,0.102609,0.0,0.0,1.0,NaN


In [4]:
def decline_by_band(F, yv, col, q=5):
    # Decline rate per quintile (numeric) or per level (categorical/flag); missing shown separately
    s = F[col]
    if s.dtype.kind in "f" and s.nunique() > 6:
        bands = pd.qcut(s, q, duplicates="drop").astype(str).replace("nan", "Missing")
    else:
        bands = s.astype(object).where(s.notna(), "Missing").astype(str)
    t = pd.DataFrame({"band": bands.values, "y": yv.values}).groupby("band", sort=False)["y"].agg(["size", "mean"])
    if s.dtype.kind in "f" and s.nunique() > 6:
        order = sorted([b for b in t.index if b != "Missing"], key=lambda b: float(b.split(",")[0].strip("([")))
        t = t.loc[order + (["Missing"] if "Missing" in t.index else [])]
    else:
        t = t.sort_index()
    return t.rename(columns={"size": "applicants", "mean": "decline_rate"})

ytr = y_train.reset_index(drop=True)
Ftr = F_train.reset_index(drop=True)
separation = []
for col in NEW_FEATURES + ["LTV", "Loan_Amount", "Net_salary", "Age"]:
    t = decline_by_band(Ftr, ytr, col)
    big = t[(t.index != "Missing") & (t["applicants"] >= 200)]["decline_rate"]
    separation.append({"feature": col, "lowest band rate": big.min(), "highest band rate": big.max(),
                       "ratio high/low": big.max() / max(big.min(), 1e-9)})
    if col in NEW_FEATURES:
        print(f"\n=== {col}")
        display(t.style.format({"decline_rate": "{:.2%}", "applicants": "{:,}"}))


=== Loan_to_Income


,applicants,decline_rate
band,,
"(0.0839, 1.906]","17,237",3.08%
"(1.906, 2.482]","17,238",4.66%
"(2.482, 3.051]","17,235",5.49%
"(3.051, 3.79]","17,237",6.20%
"(3.79, 131203.0]","17,237",6.93%



=== Vehicle_Price


,applicants,decline_rate
band,,
"(50510.982, 84182.573]","17,346",5.27%
"(84182.573, 112958.02]","17,345",5.13%
"(112958.02, 121028.937]","17,345",4.81%
"(121028.937, 147891.822]","17,345",6.42%
"(147891.822, 363111.517]","17,346",4.65%



=== Down_Payment


,applicants,decline_rate
band,,
"(378.541, 12535.873]","17,346",10.72%
"(12535.873, 18258.896]","17,345",6.66%
"(18258.896, 24850.88]","17,345",5.11%
"(24850.88, 38379.556]","17,345",2.77%
"(38379.556, 263800.517]","17,346",1.02%


... (16 outputs left out)


=== Owns_Home


,applicants,decline_rate
band,,
0.0,"4,221",4.67%
1.0,"82,494",5.29%
Missing,12,8.33%



=== Is_EV


,applicants,decline_rate
band,,
0.0,"83,944",5.35%
1.0,"2,783",2.30%


In [5]:
# One-line summary: how well does each feature separate risky applicants on its own?
sep = pd.DataFrame(separation).sort_values("ratio high/low", ascending=False).set_index("feature")
sep.style.format({"lowest band rate": "{:.2%}", "highest band rate": "{:.2%}", "ratio high/low": "{:.1f}x"})

,lowest band rate,highest band rate,ratio high/low
feature,,,
LTV,0.78%,11.97%,15.4x
Down_Payment,1.02%,10.72%,10.5x
Down_Payment_to_Income,1.20%,10.31%,8.6x
High_LTV,4.21%,14.41%,3.4x
Approx_EMI,2.62%,8.21%,3.1x
Loan_Amount,2.62%,8.21%,3.1x
Is_EV,2.30%,5.35%,2.3x
Loan_to_Income,3.08%,6.93%,2.3x
EMI_to_Income,3.08%,6.93%,2.3x


**Reading the tables:** a ratio of 1.0x means the feature tells us nothing on its own; 5x means the riskiest band declines five times as often as the safest. The **deal structure** separates risk most clearly: the less the customer puts down (high LTV, low down payment), the more often they are declined. Salary on its own barely separates risk, but salary *relative to the loan* (`EMI_to_Income`) does. This is why combined features matter.

## Step 3: Which columns to drop, and why

### 3a. Numeric redundancy (Spearman correlation)
Rule: if two numeric features have |ρ| > 0.90, they carry the same information, so we keep the one that is easier to explain.

In [6]:
NUMERIC_CANDIDATES = ["Age", "Loan_Amount", "LTV", "Net_salary", "Vehicle_Price", "Down_Payment",
                      "Approx_EMI", "Loan_to_Income", "EMI_to_Income", "Residual_Income",
                      "Down_Payment_to_Income", "tier_rank"]
rho = F_train[NUMERIC_CANDIDATES].corr(method="spearman")
display(rho.round(2).style.background_gradient(cmap="RdBu_r", vmin=-1, vmax=1))

pairs = [(a, b, rho.loc[a, b]) for i, a in enumerate(NUMERIC_CANDIDATES)
         for b in NUMERIC_CANDIDATES[i + 1:] if abs(rho.loc[a, b]) > 0.90]
pd.DataFrame(pairs, columns=["feature A", "feature B", "spearman rho"]).round(3)

,Age,Loan_Amount,LTV,Net_salary,Vehicle_Price,Down_Payment,Approx_EMI,Loan_to_Income,EMI_to_Income,Residual_Income,Down_Payment_to_Income,tier_rank
Age,1.000000,-0.180000,0.060000,0.100000,-0.240000,-0.150000,-0.180000,-0.200000,-0.200000,0.130000,-0.180000,-0.060000
Loan_Amount,-0.180000,1.000000,0.380000,0.080000,0.790000,0.020000,1.000000,0.680000,0.680000,-0.080000,-0.020000,-0.010000
LTV,0.060000,0.380000,1.000000,-0.020000,-0.150000,-0.900000,0.380000,0.310000,0.310000,-0.080000,-0.810000,-0.040000
Net_salary,0.100000,0.080000,-0.020000,1.000000,0.110000,0.070000,0.080000,-0.620000,-0.620000,0.980000,-0.340000,-0.020000
Vehicle_Price,-0.240000,0.790000,-0.150000,0.110000,1.000000,0.540000,0.790000,0.510000,0.510000,-0.020000,0.450000,0.010000
Down_Payment,-0.150000,0.020000,-0.900000,0.070000,0.540000,1.000000,0.020000,-0.040000,-0.040000,0.060000,0.890000,0.030000
Approx_EMI,-0.180000,1.000000,0.380000,0.080000,0.790000,0.020000,1.000000,0.680000,0.680000,-0.080000,-0.020000,-0.010000
Loan_to_Income,-0.200000,0.680000,0.310000,-0.620000,0.510000,-0.040000,0.680000,1.000000,1.000000,-0.740000,0.240000,0.000000
EMI_to_Income,-0.200000,0.680000,0.310000,-0.620000,0.510000,-0.040000,0.680000,1.000000,1.000000,-0.740000,0.240000,0.000000
Residual_Income,0.130000,-0.080000,-0.080000,0.980000,-0.020000,0.060000,-0.080000,-0.740000,-0.740000,1.000000,-0.340000,-0.020000


,feature A,feature B,spearman rho
0,Loan_Amount,Approx_EMI,1.000
1,Net_salary,Residual_Income,0.983
2,Loan_to_Income,EMI_to_Income,1.000


**Decisions:**
- `Approx_EMI` vs `Loan_Amount`: ρ = 1.00. With a fixed tenure and rate, EMI is just loan × a constant. **Drop `Approx_EMI`.** It still feeds `EMI_to_Income` and `Residual_Income`.
- `Loan_to_Income` vs `EMI_to_Income`: ρ = 1.00, for the same reason. **Keep `EMI_to_Income`**, because "EMI as % of salary" is the FOIR language credit teams already use. **Drop `Loan_to_Income`.**
- `Vehicle_Price` vs `Loan_Amount`: ρ ≈ 0.79, below the threshold, so **keep both**. Together with LTV they describe *how* the bike is financed.

*Alternative considered:* we could keep the raw `Loan_to_Income` and skip the EMI assumptions entirely. We chose EMI because the chatbot can show the customer the EMI, and the assumption is clearly stated.

### 3b. Categorical redundancy (Cramér's V)
Cramér's V runs from 0 (unrelated) to 1 (one field fully determines the other).

In [7]:
CATS = ["Product_Code", "Make_Code", "Model_Variant", "Model_Description", "Final_Tier", "pin_zone",
        "pin_district", "Resident_Type", "Employment_Type", "PAST_LOANS_ACTIVE"]   # full Pincode below (13k x 13k table is too big)
cv_mat = pd.DataFrame({a: {b: cramers_v(F_train[a], F_train[b]) for b in CATS} for a in CATS})
display(cv_mat.round(2).style.background_gradient(cmap="Blues", vmin=0, vmax=1))
print("Levels:", {c: F_train[c].nunique() for c in CATS}, "| Pincode:", F_train["Pincode"].nunique())
print(f"Cramer's V Pincode vs Final_Tier = {cramers_v(F_train['Pincode'], F_train['Final_Tier']):.2f}, "
      f"pin_district vs Final_Tier = {cramers_v(F_train['pin_district'], F_train['Final_Tier']):.2f}")

# Engineered flags vs the field they come from
flag_src = {"Is_First_Time_Borrower": "PAST_LOANS_ACTIVE", "Owns_Home": "Resident_Type",
            "Is_EV": "Product_Code", "Is_Stable_Income": "Employment_Type", "tier_rank": "Final_Tier"}
pd.Series({f"{k} vs {v}": cramers_v(F_train[k].astype(str), F_train[v]) for k, v in flag_src.items()},
          name="Cramer's V").round(3).to_frame()

,Product_Code,Make_Code,Model_Variant,Model_Description,Final_Tier,pin_zone,pin_district,Resident_Type,Employment_Type,PAST_LOANS_ACTIVE
Product_Code,1.000000,1.000000,1.000000,1.000000,0.150000,0.300000,0.400000,0.070000,0.150000,0.230000
Make_Code,1.000000,1.000000,1.000000,1.000000,0.110000,0.270000,0.300000,0.080000,0.100000,0.250000
Model_Variant,1.000000,1.000000,1.000000,1.000000,0.110000,0.270000,0.230000,0.080000,0.100000,0.260000
Model_Description,1.000000,1.000000,1.000000,1.000000,0.120000,0.310000,0.140000,0.080000,0.110000,0.260000
Final_Tier,0.150000,0.110000,0.110000,0.120000,1.000000,0.250000,0.630000,0.230000,0.090000,0.190000
pin_zone,0.300000,0.270000,0.270000,0.310000,0.250000,1.000000,1.000000,0.140000,0.120000,0.260000
pin_district,0.400000,0.300000,0.230000,0.140000,0.630000,1.000000,1.000000,0.300000,0.260000,0.340000
Resident_Type,0.070000,0.080000,0.080000,0.080000,0.230000,0.140000,0.300000,1.000000,0.460000,0.090000
Employment_Type,0.150000,0.100000,0.100000,0.110000,0.090000,0.120000,0.260000,0.460000,1.000000,0.160000
PAST_LOANS_ACTIVE,0.230000,0.250000,0.260000,0.260000,0.190000,0.260000,0.340000,0.090000,0.160000,1.000000


Levels: {'Product_Code': 4, 'Make_Code': 12, 'Model_Variant': 20, 'Model_Description': 86, 'Final_Tier': 7, 'pin_zone': 8, 'pin_district': 399, 'Resident_Type': 3, 'Employment_Type': 8, 'PAST_LOANS_ACTIVE': 2} | Pincode: 13330
Cramer's V Pincode vs Final_Tier = 0.92, pin_district vs Final_Tier = 0.63


,Cramer's V
Is_First_Time_Borrower vs PAST_LOANS_ACTIVE,1.0
Owns_Home vs Resident_Type,1.0
Is_EV vs Product_Code,1.0
Is_Stable_Income vs Employment_Type,1.0
tier_rank vs Final_Tier,1.0


### 3c. Vehicle fields: do we need all four?

`Model_Description` fully determines `Make_Code`, `Model_Variant` and `Product_Code` (V = 1). The open question is whether its extra detail (86 models vs 20 variants) adds signal, or just noise. We test this with 5-fold CV on train only.

In [8]:
LGBM_PARAMS = dict(n_estimators=400, learning_rate=0.03, num_leaves=31, min_child_samples=50,
                   subsample=0.8, subsample_freq=1, colsample_bytree=0.8, reg_lambda=1.0,
                   random_state=RANDOM_STATE, n_jobs=4, verbose=-1)

def make_model(kind, strategy="balanced"):
    if kind == "lgbm":
        return LGBMClassifier(**LGBM_PARAMS, class_weight="balanced" if strategy == "balanced" else None)
    return LogisticRegression(max_iter=3000, C=1.0,
                              class_weight="balanced" if strategy == "balanced" else None)

def make_pipe(feats, kind="lgbm", strategy="balanced"):
    # raw record -> clean -> domain fills -> features -> encode -> [resample] -> model
    steps = [("clean", RawCleaner()), ("domain", DomainImputer()),
             ("features", FeatureEngineer(TENURE_MONTHS, ANNUAL_RATE)),
             ("encode", build_preprocessor(**feats, scale=(kind == "lr")))]
    if strategy == "smote":
        steps.append(("resample", SMOTE(random_state=RANDOM_STATE)))
    elif strategy == "undersample":
        steps.append(("resample", RandomUnderSampler(random_state=RANDOM_STATE)))
    steps.append(("model", make_model(kind, strategy)))
    return ImbPipeline(steps)

def top_k_recall(yv, p, share=0.20):
    # Recall if we decline the riskiest `share` of applicants (threshold-free comparison)
    cut = np.quantile(p, 1 - share)
    return recall_score(yv, (p >= cut).astype(int))

def cv_eval(feats, kind="lgbm", strategy="balanced", X_=None, y_=None):
    X_ = X_train if X_ is None else X_
    y_ = y_train if y_ is None else y_
    t0 = time.time()
    oof = cross_val_predict(make_pipe(feats, kind, strategy), X_, y_, cv=CV, method="predict_proba")[:, 1]
    return {"ROC-AUC": roc_auc_score(y_, oof), "PR-AUC": average_precision_score(y_, oof),
            "Recall @0.5": recall_score(y_, (oof >= 0.5).astype(int)),
            "Recall @top20%": top_k_recall(y_, oof), "mean P(decline)": oof.mean(),
            "secs": round(time.time() - t0, 1)}, oof

# Base candidate feature set after 3a (vehicle/location fields varied below)
BASE = dict(
    skewed=["Loan_Amount", "Net_salary", "Vehicle_Price", "Down_Payment", "EMI_to_Income",
            "Down_Payment_to_Income", "Residual_Income"],
    plain=["Age", "LTV"],
    flags=["age_above_policy", "ltv_invalid", "salary_invalid", "is_non_earner", "tier_imputed",
           "Is_First_Time_Borrower", "High_LTV"],
    ordinal=["tier_rank"],
    onehot=["Qualifications", "Employment_Type", "Resident_Type", "Gender", "Age_Band", "pin_zone"],
    target_enc=["pin_district"],
)

def with_(base, **extra):
    f = {k: list(v) for k, v in base.items()}
    for k, v in extra.items():
        f[k] = f.get(k, []) + list(v)
    return f

VEHICLE_OPTIONS = {
    "A: Product + Make + Variant (one-hot)": dict(onehot=["Product_Code", "Make_Code", "Model_Variant"]),
    "B: Model_Description only (target-enc)": dict(target_enc=["Model_Description"]),
    "C: Product (one-hot) + Model_Description (target-enc)": dict(onehot=["Product_Code"], target_enc=["Model_Description"]),
}
veh = {name: cv_eval(with_(BASE, **opt))[0] for name, opt in VEHICLE_OPTIONS.items()}
veh = pd.DataFrame(veh).T
display(veh.round(4))

# Pick the simplest option within 0.002 AUC of the best (A is simplest: no target encoding)
best = veh["ROC-AUC"].max()
VEHICLE_CHOICE = next(n for n in VEHICLE_OPTIONS if veh.loc[n, "ROC-AUC"] >= best - 0.002)
print("Chosen vehicle encoding ->", VEHICLE_CHOICE)

,ROC-AUC,PR-AUC,Recall @0.5,Recall @top20%,mean P(decline),secs
A: Product + Make + Variant (one-hot),0.7781,0.1733,0.6753,0.5557,0.3361,8.5
B: Model_Description only (target-enc),0.7772,0.1730,0.6716,0.5527,0.3357,8.4
C: Product (one-hot) + Model_Description (target-enc),0.7782,0.1735,0.6711,0.5568,0.3347,8.5


Chosen vehicle encoding -> A: Product + Make + Variant (one-hot)


### 3d. Final drop list

| Column | Dropped? | Reason |
|---|---|---|
| `ID` | Dropped | Identifier, no business meaning. |
| `Final_Tier` (text) | Dropped | Replaced by `tier_rank` (V = 1, same information, keeps the order). |
| full `Pincode` (≈13k levels) | Dropped (tested in Step 4) | Too sparse. `pin_district` + `pin_zone` + `tier_rank` carry location. |
| `PAST_LOANS_ACTIVE` | Replaced | `Is_First_Time_Borrower` is the same information as a 0/1 flag (V = 1). |
| `Owns_Home`, `Is_EV`, `Is_Stable_Income` | Dropped | Pure regroupings of `Resident_Type` / `Product_Code` / `Employment_Type`, which we already one-hot, so they add nothing. |
| `Approx_EMI`, `Loan_to_Income` | Dropped | Exact duplicates of `Loan_Amount` / `EMI_to_Income` (3a). |
| Vehicle fields | See 3c | Keep the simplest set that matches the best AUC. |

### 3e. Fairness check: Gender

In [9]:
g = pd.crosstab(F_train["Gender"], y_train.values)
g.columns = ["APPROVED", "DECLINED"]
g["decline_rate"] = g["DECLINED"] / g.sum(axis=1)
p_gender = chi2_contingency(g[["APPROVED", "DECLINED"]])[1]
display(g.style.format({"decline_rate": "{:.2%}"}))
print(f"Chi-square p-value = {p_gender:.2f}  (> 0.05 means no real difference)")

,APPROVED,DECLINED,decline_rate
Gender,,,
Female,19154,1071,5.30%
Male,63015,3487,5.24%


Chi-square p-value = 0.79  (> 0.05 means no real difference)


**Should Gender be a model input?** Our recommendation is **no**, whatever the AUC:
- **Regulation and reputation:** the RBI Fair Practices Code expects lenders not to discriminate on gender. A model that uses gender is hard to defend to a regulator or the press, even when the effect is small.
- **Evidence:** decline rates are almost identical, so we expect little or no predictive loss. Step 6 measures it exactly, by training with and without Gender.
- **Practice:** we keep Gender *in the data*, only for fairness monitoring (are approval rates equal across genders after deployment?), and never as a model input.

*Alternative:* include Gender and audit the outcomes afterwards. This is rejected, because it adds legal risk for no gain.

## Step 4: Encoding and scaling inside a scikit-learn Pipeline

| Column type | Treatment | Notes |
|---|---|---|
| Nominal (few levels) | `OneHotEncoder(handle_unknown="infrequent_if_exist", min_frequency=30)` | Levels with < 30 rows are grouped. **Unseen** levels (a new bike model) go into that "infrequent" bucket, or are ignored, so nothing breaks. This is the safer form of `handle_unknown="ignore"`. |
| High-cardinality (`pin_district`, 399 levels) | Compared below: one-hot vs **TargetEncoder** | sklearn's `TargetEncoder` uses internal 5-fold cross-fitting, so a row's own Decision never feeds its own encoding. It is also refitted inside every CV fold, because it lives in the pipeline. |
| Ordinal (`tier_rank`, 1-7) | Kept as a number | Preserves the Megapolis → Rural order. |
| Skewed money fields and ratios | Cap at 0.5% / 99.5% → **log** → median-impute (+ "was missing" flag) | Log compresses the long right tail (a few very high salaries). |
| Scaling | `StandardScaler` **only for logistic regression** | LR's penalty and solver assume features on similar scales. Trees split on rank order, so scaling (and even log) changes nothing for LightGBM. We skip scaling there to keep the pipeline simpler and faster. |

In [10]:
FEATS = with_(BASE, **VEHICLE_OPTIONS[VEHICLE_CHOICE])

ENCODING_OPTIONS = {
    "pin_district target-encoded": FEATS,
    "pin_district one-hot (min_freq=30)": {**FEATS, "target_enc": [c for c in FEATS["target_enc"] if c != "pin_district"],
                                           "onehot": FEATS["onehot"] + ["pin_district"]},
    "+ full Pincode target-encoded": with_(FEATS, target_enc=["Pincode"]),
}
rows = {}
for name, f in ENCODING_OPTIONS.items():
    for kind in ["lgbm", "lr"]:
        rows[(name, kind)] = cv_eval(f, kind)[0]
enc = pd.DataFrame(rows).T
display(enc.round(4))

best_lgbm = enc.xs("lgbm", level=1)["ROC-AUC"]
ENC_CHOICE = next(n for n in ENCODING_OPTIONS if best_lgbm[n] >= best_lgbm.max() - 0.002)
FEATS = ENCODING_OPTIONS[ENC_CHOICE]
print("Chosen encoding ->", ENC_CHOICE)

ROC-AUC  PR-AUC  Recall @0.5  Recall @top20%  mean P(decline)  secs
pin_district target-encoded        lgbm   0.7781  0.1733       0.6753          0.5557           0.3361   9.1
                                   lr     0.7773  0.1700       0.7334          0.5502           0.3945   4.7
pin_district one-hot (min_freq=30) lgbm   0.7813  0.1744       0.6948          0.5610           0.3441  11.1
                                   lr     0.7775  0.1701       0.7253          0.5520           0.3765  15.3
+ full Pincode target-encoded      lgbm   0.7797  0.1755       0.6738          0.5614           0.3349  11.7
                                   lr     0.7780  0.1701       0.7348          0.5538           0.3943   5.3

Chosen encoding -> pin_district one-hot (min_freq=30)


In [11]:
# Final number of model columns after encoding (fitted on train only)
for kind in ["lgbm", "lr"]:
    enc_pipe = processing_pipeline(build_preprocessor(**FEATS, scale=(kind == "lr")), TENURE_MONTHS, ANNUAL_RATE)
    Xt = enc_pipe.fit_transform(X_train, y_train)
    print(f"{kind.upper():5s}: {sum(len(v) for v in FEATS.values())} input features -> {Xt.shape[1]} model columns "
          f"| train matrix {Xt.shape}")
print({k: v for k, v in FEATS.items()})

LGBM : 27 input features -> 446 model columns | train matrix (86727, 446)


LR   : 27 input features -> 446 model columns | train matrix (86727, 446)
{'skewed': ['Loan_Amount', 'Net_salary', 'Vehicle_Price', 'Down_Payment', 'EMI_to_Income', 'Down_Payment_to_Income', 'Residual_Income'], 'plain': ['Age', 'LTV'], 'flags': ['age_above_policy', 'ltv_invalid', 'salary_invalid', 'is_non_earner', 'tier_imputed', 'Is_First_Time_Borrower', 'High_LTV'], 'ordinal': ['tier_rank'], 'onehot': ['Qualifications', 'Employment_Type', 'Resident_Type', 'Gender', 'Age_Band', 'pin_zone', 'Product_Code', 'Make_Code', 'Model_Variant', 'pin_district'], 'target_enc': []}


## Step 5: Imbalance handling (compare, don't assume)

Every option is run inside the **same** 5-fold CV on train. SMOTE and undersampling are steps **inside** the `imblearn` pipeline, so they only ever touch the training folds. The scored fold is always real, untouched data.

We report:
- **ROC-AUC**: how well the model ranks risk.
- **PR-AUC**: precision-recall, which focuses on the declines.
- **Recall @0.5**: declines caught at the default cut-off.
- **Recall @top20%**: declines caught if we decline the 20% riskiest applicants. This is threshold-free and comparable across strategies.

In [12]:
STRATEGIES = {"a) no adjustment": "none", "b) class_weight='balanced'": "balanced",
              "c1) SMOTE (in-fold)": "smote", "c2) random undersampling (in-fold)": "undersample"}
rows, OOF = {}, {}
for kind in ["lgbm", "lr"]:
    for name, s in STRATEGIES.items():
        res, oof = cv_eval(FEATS, kind, s)
        rows[(kind, name)] = res
        OOF[(kind, s)] = oof
imb = pd.DataFrame(rows).T
imb["actual decline rate"] = y_train.mean()
imb.round(4)

ROC-AUC  PR-AUC  Recall @0.5  Recall @top20%  mean P(decline)  secs  actual decline rate
lgbm a) no adjustment                     0.7818  0.1754       0.0013          0.5627           0.0520  14.0               0.0526
     b) class_weight='balanced'           0.7813  0.1744       0.6948          0.5610           0.3441  16.7               0.0526
     c1) SMOTE (in-fold)                  0.7750  0.1619       0.0191          0.5445           0.0681  42.0               0.0526
     c2) random undersampling (in-fold)   0.7595  0.1476       0.7264          0.5165           0.3955   9.8               0.0526
lr   a) no adjustment                     0.7802  0.1733       0.0061          0.5627           0.0526  10.9               0.0526
     b) class_weight='balanced'           0.7775  0.1701       0.7253          0.5520           0.3765  17.3               0.0526
     c1) SMOTE (in-fold)                  0.7737  0.1684       0.7016          0.5459           0.3588  34.4               0.0526
     c2) random undersampling (in-fold)   0.7740  0.1624       0.7282          0.5452           0.3902   5.2               0.0526

**How to read this:**
- **No strategy improves the ranking.** With no adjustment and with balanced weights, LightGBM reaches the same CV AUC (≈0.781). SMOTE is slightly worse. **Undersampling is clearly worse for LightGBM** (≈0.760), because it throws away about 90% of the approvals. Resampling does not make the model smarter.
- **Recall @0.5** is near zero with no adjustment. The model rarely predicts above 50% when only 5% of cases are declines, so this is a **threshold problem, not a model problem**.
  - Balanced weights and undersampling fix it.
  - SMOTE does **not** fix it for LightGBM: mean P(decline) only rises to about 7%. The synthetic "in-between" applicants it creates in one-hot space are largely ignored by the trees.
- **Calibration:** compare "mean P(decline)" with the true 5.3%.
  - Balanced weights and undersampling push the average to about 34-40%, so their scores are **risk scores, not real probabilities**. If the business needs a true probability (for risk-based pricing or expected-loss estimates), recalibrate (Platt / isotonic on held-out data).
  - The unweighted model is already calibrated (5.2%).

**Recommendation: LightGBM with `class_weight="balanced"`.**
- It ties for the best ranking.
- It works with the familiar 0.5 cut-off out of the box.
- It doesn't invent applicants (SMOTE) or discard real ones (undersampling).
- It is the simplest to explain and reproduce.

*Equally valid alternative:* no adjustment plus a tuned threshold. It has the same ranking and keeps calibrated probabilities; choose it if expected-loss pricing is planned. Either way, the approve/decline **cut-off** is chosen separately in Step 6.

In [13]:
best_by_strategy = imb.xs("lgbm", level=0)["ROC-AUC"]
MODEL_KIND = "lgbm" if imb.xs("lgbm", level=0)["ROC-AUC"].max() >= imb.xs("lr", level=0)["ROC-AUC"].max() else "lr"
STRATEGY = "balanced"
print(f"Model family: {MODEL_KIND} | imbalance strategy: {STRATEGY}")

Model family: lgbm | imbalance strategy: balanced


## Step 6: Feature selection for the minimal question set

### 6a. Fit both models on train; check on validation
The cut-off (threshold) is chosen on the **train out-of-fold** predictions, never on validation, by maximising **F2**. F2 weights recall twice as much as precision, matching "catching declines matters most".

*Alternative:* fix a target recall (for example 70%) and accept whatever share of applicants gets declined. The trade-off table below lets the business choose.

In [14]:
def f2_threshold(yv, p):
    grid = np.unique(np.quantile(p, np.linspace(0.50, 0.995, 200)))
    scores = [fbeta_score(yv, (p >= t).astype(int), beta=2) for t in grid]
    return float(grid[int(np.argmax(scores))])

def val_report(pipe, thr, X_=None, y_=None):
    X_ = X_val if X_ is None else X_
    y_ = y_val if y_ is None else y_
    p = pipe.predict_proba(X_)[:, 1]
    pred = (p >= thr).astype(int)
    return {"ROC-AUC": roc_auc_score(y_, p), "PR-AUC": average_precision_score(y_, p),
            "threshold": thr, "Recall (DECLINED)": recall_score(y_, pred),
            "Precision (DECLINED)": precision_score(y_, pred, zero_division=0),
            "% applicants declined": pred.mean()}, p

fitted, thresholds, val_rows, VAL_P = {}, {}, {}, {}
for kind in ["lgbm", "lr"]:
    thresholds[kind] = f2_threshold(y_train, OOF[(kind, STRATEGY)])
    fitted[kind] = make_pipe(FEATS, kind, STRATEGY).fit(X_train, y_train)
    val_rows[kind], VAL_P[kind] = val_report(fitted[kind], thresholds[kind])
pd.DataFrame(val_rows).T.round(4)

,ROC-AUC,PR-AUC,threshold,Recall (DECLINED),Precision (DECLINED),% applicants declined
lgbm,0.7907,0.1857,0.5173,0.6886,0.1260,0.2872
lr,0.7856,0.1739,0.5510,0.6798,0.1276,0.2802


In [15]:
# Business trade-off on validation: what recall costs in declined applicants (chosen model)
p = VAL_P[MODEL_KIND]
rows = []
for target in [0.5, 0.6, 0.7, 0.8, 0.9]:
    oof = OOF[(MODEL_KIND, STRATEGY)]
    t = np.quantile(oof[y_train.values == 1], 1 - target)    # threshold from TRAIN out-of-fold scores
    pred = (p >= t).astype(int)
    rows.append({"target recall": f"{target:.0%}", "threshold": round(t, 3),
                 "val recall": recall_score(y_val, pred), "val precision": precision_score(y_val, pred),
                 "% applicants declined": pred.mean()})
pd.DataFrame(rows).style.format({"val recall": "{:.1%}", "val precision": "{:.1%}", "% applicants declined": "{:.1%}"})

,target recall,threshold,val recall,val precision,% applicants declined
0,50%,0.629000,51.3%,15.9%,17.0%
1,60%,0.566000,61.8%,13.9%,23.4%
2,70%,0.496000,71.8%,12.2%,30.9%
3,80%,0.389000,84.4%,10.4%,42.6%
4,90%,0.253000,91.3%,8.3%,57.7%


### 6b. Feature importance, grouped back to the chatbot question

In [16]:
# Which source does each model feature come from? (customer question / dealer quote / catalogue)
FEATURE_SOURCE = {
    "Age": "Age", "Age_Band": "Age", "age_above_policy": "Age",
    "Net_salary": "Net_salary", "EMI_to_Income": "Net_salary", "Residual_Income": "Net_salary",
    "Down_Payment_to_Income": "Net_salary", "salary_invalid": "Net_salary",
    "Employment_Type": "Employment_Type", "is_non_earner": "Employment_Type",
    "Qualifications": "Qualifications", "Resident_Type": "Resident_Type", "Gender": "Gender",
    "Is_First_Time_Borrower": "PAST_LOANS_ACTIVE",
    "pin_district": "Pincode", "pin_zone": "Pincode", "Pincode": "Pincode",
    "tier_rank": "Pincode", "tier_imputed": "Pincode",
    "Loan_Amount": "Dealer quote", "LTV": "Dealer quote", "Vehicle_Price": "Dealer quote",
    "Down_Payment": "Dealer quote", "High_LTV": "Dealer quote", "ltv_invalid": "Dealer quote",
    "Product_Code": "Catalogue", "Make_Code": "Catalogue", "Model_Variant": "Catalogue",
    "Model_Description": "Catalogue",
}
# Features that also NEED a second customer answer (removed in ablation if either is missing)
EXTRA_DEPENDENCY = {"salary_invalid": "Employment_Type", "is_non_earner": "Employment_Type"}

def source_of(model_col):
    name = model_col.split("__", 1)[1].replace("missingindicator_", "")
    match = max((f for f in FEATURE_SOURCE if name == f or name.startswith(f + "_")), key=len)
    return match, FEATURE_SOURCE[match]

lgbm_pipe = fitted["lgbm"]
cols = lgbm_pipe.named_steps["encode"].get_feature_names_out()
gain = lgbm_pipe.named_steps["model"].booster_.feature_importance(importance_type="gain")
gi = pd.DataFrame({"model_column": cols, "gain": gain})
gi[["feature", "question"]] = [source_of(c) for c in cols]
gain_by_q = (gi.groupby("question")["gain"].sum() / gi["gain"].sum()).sort_values(ascending=False)
display(gi.groupby("feature")["gain"].sum().div(gi["gain"].sum()).sort_values(ascending=False)
          .head(15).to_frame("share of LightGBM gain").style.format("{:.1%}"))
gain_by_q.to_frame("share of LightGBM gain").style.format("{:.1%}")

,share of LightGBM gain
feature,
LTV,34.2%
pin_district,18.6%
Down_Payment,5.9%
Loan_Amount,4.7%
Vehicle_Price,4.5%
Age,4.2%
Down_Payment_to_Income,4.1%
pin_zone,4.0%
EMI_to_Income,3.3%


,share of LightGBM gain
question,
Dealer quote,49.4%
Pincode,23.2%
Net_salary,11.8%
Catalogue,5.2%
Age,4.9%
PAST_LOANS_ACTIVE,1.9%
Qualifications,1.7%
Employment_Type,1.1%
Gender,0.6%


In [17]:
# Permutation importance on VALIDATION, at the raw-input level: shuffle one raw field and
# measure the AUC loss. This automatically includes every feature derived from that field.
RAW_TO_GROUP = {c: (c if c in cp.CUSTOMER_QUESTIONS else
                    "Pincode" if c == "Final_Tier" else
                    "Dealer quote" if c in cp.DEAL_INPUTS else "Catalogue") for c in X_val.columns}
perm = {}
for kind in ["lgbm", "lr"]:
    r = permutation_importance(fitted[kind], X_val, y_val, scoring="roc_auc", n_repeats=5,
                               random_state=RANDOM_STATE, n_jobs=1)
    perm[kind] = pd.Series(r.importances_mean, index=X_val.columns)
perm = pd.DataFrame(perm)
perm["group"] = perm.index.map(RAW_TO_GROUP)
perm_q = perm.groupby("group")[["lgbm", "lr"]].sum().sort_values("lgbm", ascending=False)
perm_q.columns = ["AUC loss when shuffled: LightGBM", "AUC loss when shuffled: LogReg"]
perm_q.round(4)

,AUC loss when shuffled: LightGBM,AUC loss when shuffled: LogReg
group,,
Dealer quote,0.2130,0.2006
Pincode,0.0445,0.0695
PAST_LOANS_ACTIVE,0.0073,0.0044
Catalogue,0.0065,0.0174
Age,0.0059,0.0039
Employment_Type,0.0041,0.0031
Net_salary,0.0033,0.0006
Qualifications,0.0023,0.0015
Gender,0.0006,-0.0000


### 6c. Ablation: remove one customer question at a time

For each question, we drop **every** feature that depends on it, retrain, and measure:
- the 5-fold **CV AUC on train**;
- the **validation AUC**;
- **validation recall at a fixed 25% decline rate**. Every configuration declines the same share of applicants, so recall is compared like-for-like. Recall at each configuration's own F2 threshold would mix "better model" with "declines more people".

A note on noise: a single AUC on about 1,100 validation declines has a standard error of roughly ±0.008. But comparing two models **on the same validation rows** (a paired comparison) is far more precise, so a validation loss of 0.005 or more is a meaningful warning sign.

In [18]:
def drop_questions(feats, questions):
    questions = set(questions)
    def keep(f):
        return FEATURE_SOURCE.get(f) not in questions and EXTRA_DEPENDENCY.get(f) not in questions
    return {k: [f for f in v if keep(f)] for k, v in feats.items()}

FIXED_DECLINE_SHARE = 0.25     # compare recall at the same share of applicants declined

def run_config(feats, label):
    cvres, oof = cv_eval(feats, MODEL_KIND, STRATEGY)
    pipe = make_pipe(feats, MODEL_KIND, STRATEGY).fit(X_train, y_train)
    p = pipe.predict_proba(X_val)[:, 1]
    return {"config": label, "CV AUC": cvres["ROC-AUC"], "CV PR-AUC": cvres["PR-AUC"],
            "Val AUC": roc_auc_score(y_val, p), "Val PR-AUC": average_precision_score(y_val, p),
            "Val recall": top_k_recall(y_val, p, FIXED_DECLINE_SHARE)}

t0 = time.time()
full = run_config(FEATS, "ALL questions")
abl = [full]
for q in cp.CUSTOMER_QUESTIONS:
    abl.append(run_config(drop_questions(FEATS, [q]), f"without {q}"))
abl = pd.DataFrame(abl).set_index("config")
abl["CV AUC drop"] = full["CV AUC"] - abl["CV AUC"]
abl["Val AUC drop"] = full["Val AUC"] - abl["Val AUC"]
abl["Val recall change"] = abl["Val recall"] - full["Val recall"]
abl = abl.rename(columns={"Val recall": f"Val recall @{FIXED_DECLINE_SHARE:.0%} declined"})
print(f"ablation took {time.time() - t0:.0f}s")
abl.round(4)

ablation took 161s


,CV AUC,CV PR-AUC,Val AUC,Val PR-AUC,Val recall @25% declined,CV AUC drop,Val AUC drop,Val recall change
config,,,,,,,,
ALL questions,0.7813,0.1744,0.7907,0.1857,0.6342,0.0000,0.0000,0.0000
without Qualifications,0.7809,0.1741,0.7888,0.1861,0.6333,0.0005,0.0019,-0.0009
without Employment_Type,0.7805,0.1744,0.7881,0.1844,0.6342,0.0008,0.0026,0.0000
without Age,0.7804,0.1733,0.7872,0.1797,0.6325,0.0010,0.0035,-0.0018
without Pincode,0.7576,0.1519,0.7685,0.1632,0.6044,0.0237,0.0222,-0.0298
without Gender,0.7812,0.1756,0.7891,0.1820,0.6404,0.0002,0.0016,0.0061
without Resident_Type,0.7819,0.1755,0.7894,0.1848,0.6412,-0.0006,0.0013,0.0070
without Net_salary,0.7830,0.1758,0.7909,0.1871,0.6360,-0.0017,-0.0002,0.0018
without PAST_LOANS_ACTIVE,0.7786,0.1704,0.7860,0.1808,0.6298,0.0028,0.0047,-0.0044


### 6d. From single removals to a combined question set

Removing questions one at a time can hide a combined loss: two questions might each cost 0.003, but 0.006 together. So we remove the cheapest questions **one after another** (greedy backward elimination). A question is removed only if the **combined** loss versus the all-questions model stays below 0.005 on **both** CV and validation.

*Why both:* in a first run that used CV alone, the greedy search removed every question except Pincode. CV said the loss was 0.0049, while validation showed 0.012. Requiring both is the more conservative and more defensible choice.

**Gender is removed by policy** (Step 3e), whatever its AUC contribution.

In [19]:
AUC_TOLERANCE = 0.005
single_drop = abl["CV AUC drop"].drop("ALL questions")
single_drop.index = single_drop.index.str.replace("without ", "", regex=False)

removed = ["Gender"]                                   # policy decision
current = run_config(drop_questions(FEATS, removed), "policy: no Gender")
KEYS = ["CV AUC", "Val AUC", "Val recall"]
history = [{"step": "start (all questions)", **{k: full[k] for k in KEYS}},
           {"step": "remove Gender (policy)", **{k: current[k] for k in KEYS}}]
blocked = {}
single_val = abl["Val AUC drop"].drop("ALL questions")
single_val.index = single_val.index.str.replace("without ", "", regex=False)
for q in single_drop.sort_values().index:
    if q in removed:
        continue
    if single_drop[q] >= AUC_TOLERANCE or single_val[q] >= AUC_TOLERANCE:
        blocked[q] = "alone"
        continue
    trial = run_config(drop_questions(FEATS, removed + [q]), f"- {q}")
    cv_loss, val_loss = full["CV AUC"] - trial["CV AUC"], full["Val AUC"] - trial["Val AUC"]
    if cv_loss < AUC_TOLERANCE and val_loss < AUC_TOLERANCE:
        removed.append(q)
        current = trial
        history.append({"step": f"remove {q}", **{k: trial[k] for k in KEYS}})
    else:
        blocked[q] = f"combined loss CV {cv_loss:.4f} / val {val_loss:.4f}"
        history.append({"step": f"keep {q} (combined loss CV {cv_loss:.4f}, val {val_loss:.4f})",
                        **{k: trial[k] for k in KEYS}})
history = pd.DataFrame(history)
history["CV AUC loss vs all"] = full["CV AUC"] - history["CV AUC"]
history["Val AUC loss vs all"] = full["Val AUC"] - history["Val AUC"]
history = history.rename(columns={"Val recall": f"Val recall @{FIXED_DECLINE_SHARE:.0%} declined"})
display(history.round(4))
KEEP_QUESTIONS = [q for q in cp.CUSTOMER_QUESTIONS if q not in removed]
print("Questions removed:", removed)
print("Questions kept   :", KEEP_QUESTIONS)

,step,CV AUC,Val AUC,Val recall @25% declined,CV AUC loss vs all,Val AUC loss vs all
0,start (all questions),0.7813,0.7907,0.6342,0.0000,0.0000
1,remove Gender (policy),0.7812,0.7891,0.6404,0.0002,0.0016
2,remove Net_salary,0.7824,0.7902,0.6368,-0.0011,0.0005
3,remove Resident_Type,0.7833,0.7897,0.6377,-0.0020,0.0010
4,remove Qualifications,0.7817,0.7884,0.6412,-0.0003,0.0023
5,remove Employment_Type,0.7815,0.7858,0.6281,-0.0002,0.0049
6,"keep Age (combined loss CV 0.0017, val 0.0074)",0.7797,0.7833,0.6219,0.0017,0.0074
7,keep PAST_LOANS_ACTIVE (combined loss CV 0.003...,0.7778,0.7817,0.6246,0.0035,0.0090


Questions removed: ['Gender', 'Net_salary', 'Resident_Type', 'Qualifications', 'Employment_Type']
Questions kept   : ['Age', 'Pincode', 'PAST_LOANS_ACTIVE']


In [20]:
# The minimal-question-set table
def feeds(q):
    return ", ".join(sorted({f for f, s in FEATURE_SOURCE.items()
                             if (s == q or EXTRA_DEPENDENCY.get(f) == q) and any(f in v for v in FEATS.values())}))

def reason(q):
    d = single_drop[q]
    if q == "Gender":
        return f"Removed by fairness policy (cost only {d:+.4f} AUC; decline rates equal across genders)"
    v = single_val[q]
    if q in removed:
        return f"Costs < {AUC_TOLERANCE} AUC on CV and validation, alone and combined: not worth a question"
    if blocked.get(q) == "alone":
        return f"Removal alone costs {max(d, v):.4f} AUC (>= {AUC_TOLERANCE})"
    return f"Cheap alone, but removing it on top of the others: {blocked.get(q)} (>= {AUC_TOLERANCE})"

QUESTION_TABLE = pd.DataFrame([{
    "Question": q, "Features it feeds": feeds(q),
    "AUC drop if removed (CV)": round(single_drop[q], 4),
    "AUC drop if removed (val)": round(abl.loc[f"without {q}", "Val AUC drop"], 4),
    f"Recall change if removed (val, @{FIXED_DECLINE_SHARE:.0%} declined)": f"{abl.loc[f'without {q}', 'Val recall change']:+.1%}",
    "Keep?": "Yes" if q in KEEP_QUESTIONS else "No", "Reason": reason(q),
} for q in cp.CUSTOMER_QUESTIONS]).sort_values("AUC drop if removed (val)", ascending=False)
QUESTION_TABLE.to_csv(f"{ART}/minimal_question_set.csv", index=False)
QUESTION_TABLE

,Question,Features it feeds,AUC drop if removed (CV),AUC drop if removed (val),"Recall change if removed (val, @25% declined)",Keep?,Reason
3,Pincode,"pin_district, pin_zone, tier_imputed, tier_rank",0.0237,0.0222,-3.0%,Yes,Removal alone costs 0.0237 AUC (>= 0.005)
7,PAST_LOANS_ACTIVE,Is_First_Time_Borrower,0.0028,0.0047,-0.4%,Yes,"Cheap alone, but removing it on top of the oth..."
2,Age,"Age, Age_Band, age_above_policy",0.0010,0.0035,-0.2%,Yes,"Cheap alone, but removing it on top of the oth..."
1,Employment_Type,"Employment_Type, is_non_earner, salary_invalid",0.0008,0.0026,+0.0%,No,"Costs < 0.005 AUC on CV and validation, alone ..."
0,Qualifications,Qualifications,0.0005,0.0019,-0.1%,No,"Costs < 0.005 AUC on CV and validation, alone ..."
4,Gender,Gender,0.0002,0.0016,+0.6%,No,Removed by fairness policy (cost only +0.0002 ...
5,Resident_Type,Resident_Type,-0.0006,0.0013,+0.7%,No,"Costs < 0.005 AUC on CV and validation, alone ..."
6,Net_salary,"Down_Payment_to_Income, EMI_to_Income, Net_sal...",-0.0017,-0.0002,+0.2%,No,"Costs < 0.005 AUC on CV and validation, alone ..."


### 6e. Business caveats on the question set
- **Net_salary adds nothing to the model.** Self-reported, unverified income is weak evidence, and the deal structure (LTV, down payment) already captures affordability. **But** credit policy or regulation (for example a FOIR check, or RBI digital-lending norms on assessing repayment capacity) may still require us to *collect* income. If so, ask it for compliance, not for the score.
- **Age** is kept by the model, and is needed anyway for the 18-60 eligibility rule and KYC.
- **Pincode** is by far the most valuable customer question. It also drives the tier lookup, so it should be asked early in the chat.
- The tolerance (0.005 AUC) is a business choice. A looser tolerance would also drop Age and past loans, at a measurable cost on validation (combined loss 0.009).

## Step 7: Package for production

The final configuration is: chosen model + `class_weight="balanced"` + only the kept questions, plus the dealer quote, the catalogue and the tier lookup.

Everything (cleaning → business-aware fills → feature engineering → encoding) is **one** sklearn `Pipeline`, fitted on **train only**. We save two files:
- `processing_pipeline.joblib`: raw record → model-ready features (what the brief asks for).
- `approval_model.joblib`: the full pipeline including the model, plus the decision threshold and the list of required questions. This is what the chatbot calls.

In [21]:
FINAL_FEATS = drop_questions(FEATS, removed)
FINAL_RAW_INPUTS = KEEP_QUESTIONS + cp.DEAL_INPUTS + cp.CATALOGUE_INPUTS + ["Final_Tier"]

# Threshold from train out-of-fold predictions of the FINAL configuration
_, oof_final = cv_eval(FINAL_FEATS, MODEL_KIND, STRATEGY)
FINAL_THRESHOLD = f2_threshold(y_train, oof_final)

processing = processing_pipeline(build_preprocessor(**FINAL_FEATS, scale=(MODEL_KIND == "lr")),
                                 TENURE_MONTHS, ANNUAL_RATE)
Xt_train = processing.fit_transform(X_train[FINAL_RAW_INPUTS], y_train)
Xt_val = processing.transform(X_val[FINAL_RAW_INPUTS])
model = make_model(MODEL_KIND, STRATEGY).fit(Xt_train, y_train)

final_pipe = ImbPipeline(processing.steps + [("model", model)])     # already fitted steps
final_val, p_final = val_report(final_pipe, FINAL_THRESHOLD, X_val[FINAL_RAW_INPUTS])
print("Final model on validation:")
display(pd.Series(final_val).round(4).to_frame("value"))

joblib.dump(processing, f"{ART}/processing_pipeline.joblib")
joblib.dump({"pipeline": final_pipe, "threshold": FINAL_THRESHOLD, "model_kind": MODEL_KIND,
             "required_inputs": FINAL_RAW_INPUTS, "customer_questions": KEEP_QUESTIONS,
             "emi_assumption": {"tenure_months": TENURE_MONTHS, "annual_rate": ANNUAL_RATE},
             "trained_on_rows": len(X_train), "random_state": RANDOM_STATE},
            f"{ART}/approval_model.joblib")

# Deliverable 2: processed train / validation datasets
names = processing.named_steps["encode"].get_feature_names_out()
pd.DataFrame(Xt_train, columns=names).assign(y=y_train.values).to_csv(f"{ART}/processed_train.csv.gz", index=False)
pd.DataFrame(Xt_val, columns=names).assign(y=y_val.values).to_csv(f"{ART}/processed_val.csv.gz", index=False)
print("processed train:", Xt_train.shape, "| processed val:", Xt_val.shape)
print("saved:", sorted(os.listdir(ART)))

Final model on validation:


,value
ROC-AUC,0.7858
PR-AUC,0.1832
threshold,0.5756
Recall (DECLINED),0.6044
Precision (DECLINED),0.1362
% applicants declined,0.2333


processed train: (86727, 415) | processed val: (21682, 415)
saved: ['approval_model.joblib', 'minimal_question_set.csv', 'processed_train.csv.gz', 'processed_val.csv.gz', 'processing_pipeline.joblib']


In [22]:
# ---- Production tests: reload from disk exactly as the chatbot service would ----
bundle = joblib.load(f"{ART}/approval_model.joblib")
proc = joblib.load(f"{ART}/processing_pipeline.joblib")
pipe, thr = bundle["pipeline"], bundle["threshold"]

def decide(record: dict) -> dict:
    p = float(pipe.predict_proba(pd.DataFrame([record]))[0, 1])
    return {"P(decline)": round(p, 4), "decision": "DECLINED" if p >= thr else "APPROVED"}

# (a) whole validation set
Xa = proc.transform(X_val[FINAL_RAW_INPUTS])
print("(a) validation transform:", Xa.shape, "| NaN:", int(np.isnan(Xa).sum()))

# (b) one raw applicant, exactly as the chatbot would collect it (strings, like a web form)
applicant = {"Qualifications": "GRAD", "Employment_Type": "SAL", "Age": "29", "Pincode": "560068",
             "Gender": "Female", "Resident_Type": "O", "Net_salary": "32000", "PAST_LOANS_ACTIVE": "NO_PAST_LOANS",
             "Loan_Amount": "95000", "LTV": "82.5", "Product_Code": "SC", "Make_Code": "JUPITER",
             "Model_Description": "JUPITER 125 BSVI - DISC", "Model_Variant": "125 CC DISC",
             "Final_Tier": "02 Megapolis (B)"}
applicant = {k: v for k, v in applicant.items() if k in FINAL_RAW_INPUTS}   # only what we ask
print("(b) single applicant:", decide(applicant))

# (c) unseen Make_Code, a blank field, Age = 65, and a missing (never-asked) key
edge = dict(applicant, Make_Code="HONDA", Model_Description="HONDA ACTIVA 6G", Model_Variant="110 CC",
            Age=65, Net_salary="")
edge.pop("Final_Tier")                         # lookup failed -> tier filled from pincode district
print("(c) edge case       :", decide(edge))
print("    model-ready row OK:", proc.transform(pd.DataFrame([edge])).shape)

(a) validation transform: (21682, 415) | NaN: 0
(b) single applicant: {'P(decline)': 0.4226, 'decision': 'APPROVED'}
(c) edge case       : {'P(decline)': 0.5504, 'decision': 'APPROVED'}
    model-ready row OK: (1, 415)


In [23]:
# ---- Latency: single-record transform + predict (target < 2 seconds) ----
rec = pd.DataFrame([applicant])
for _ in range(5):
    pipe.predict_proba(rec)                     # warm-up
times = []
for _ in range(200):
    t0 = time.perf_counter()
    pipe.predict_proba(rec)
    times.append((time.perf_counter() - t0) * 1000)
print(f"Single applicant, clean + features + encode + predict: median {np.median(times):.1f} ms, "
      f"95th percentile {np.percentile(times, 95):.1f} ms, max {np.max(times):.1f} ms (target 2000 ms)")

Single applicant, clean + features + encode + predict: median 15.1 ms, 95th percentile 17.0 ms, max 23.4 ms (target 2000 ms)


## Deliverable 4: Feature Engineering Log

In [24]:
in_final = lambda f: any(f in v for v in FINAL_FEATS.values())
LOG = [
    ("Loan_to_Income", "Loan_Amount, Net_salary", "Loan ÷ monthly salary (NaN if salary ≤ 0)", "Affordability: months of salary borrowed", "No: identical ranking to EMI_to_Income (ρ = 1)"),
    ("Vehicle_Price", "Loan_Amount, LTV", "Loan ÷ (LTV/100)", "Price of bike relative to profile", None),
    ("Down_Payment", "Vehicle_Price, Loan_Amount", "Price − Loan (≥ 0)", "Customer's own money in the deal; strongest single risk signal", None),
    ("Approx_EMI", "Loan_Amount", f"Reducing-balance EMI, {TENURE_MONTHS} months @ {ANNUAL_RATE:.0%} p.a.", "Monthly obligation", "No: equals Loan_Amount × constant (ρ = 1)"),
    ("EMI_to_Income", "Approx_EMI, Net_salary", "EMI ÷ salary", "FOIR-style repayment burden", None),
    ("Residual_Income", "Net_salary, Approx_EMI", "Salary − EMI", "Money left to live on", None),
    ("Down_Payment_to_Income", "Down_Payment, Net_salary", "Down payment ÷ salary", "Savings capacity", None),
    ("Age_Band", "Age", "18-25 / 26-35 / 36-45 / 46-60 / 60+", "Non-linear age effect for LogReg", None),
    ("Is_First_Time_Borrower", "PAST_LOANS_ACTIVE", "1 if NO_PAST_LOANS", "Thin credit file", None),
    ("High_LTV", "LTV", "1 if LTV ≥ 90%", "Near-zero down payment red flag", None),
    ("Is_Stable_Income", "Employment_Type", "1 if SAL / PEN", "Income stability", "No: regrouping of one-hot Employment_Type"),
    ("Owns_Home", "Resident_Type", "1 if owned", "Residential stability", "No: regrouping of one-hot Resident_Type"),
    ("Is_EV", "Product_Code", "1 if EB", "EV segment", "No: regrouping of Product_Code"),
    ("tier_rank", "Final_Tier / Pincode", "Tier number 1-7; blanks from same 3-digit pincode", "City size, ordinal", None),
    ("pin_district", "Pincode", "First 3 digits; one-hot with min_frequency=30 (beat target encoding and full pincode in Step 4)", "Local risk without 13k sparse codes", None),
    ("pin_zone", "Pincode", "First digit, one-hot", "Broad region", None),
    ("age_above_policy", "Age", "1 if Age > 60", "Outside 18-60 policy", None),
    ("salary_invalid / is_non_earner", "Net_salary, Employment_Type", "Typo salary / student or non-earner", "Data-quality and income-type signals", None),
    ("ltv_invalid / tier_imputed", "LTV / Final_Tier", "1 if value was rebuilt", "Data-quality signals", None),
]
def kept(name, override):
    if override:
        return override
    parts = [p.strip() for p in name.split("/")]
    ok = all(in_final(p) for p in parts)
    if ok:
        return "Yes"
    src = {FEATURE_SOURCE.get(p) for p in parts}
    return f"No: depends on a question not asked ({', '.join(sorted(s for s in src if s in removed))})" \
        if src & set(removed) else "No"
FE_LOG = pd.DataFrame([{"Feature": f, "Created from": src, "Formula / method": how, "Why it helps": why,
                        "Kept?": kept(f, over)} for f, src, how, why, over in LOG])
FE_LOG.to_csv(f"{ART}/feature_engineering_log.csv", index=False)
FE_LOG

,Feature,Created from,Formula / method,Why it helps,Kept?
0,Loan_to_Income,"Loan_Amount, Net_salary",Loan ÷ monthly salary (NaN if salary ≤ 0),Affordability: months of salary borrowed,No: identical ranking to EMI_to_Income (ρ = 1)
1,Vehicle_Price,"Loan_Amount, LTV",Loan ÷ (LTV/100),Price of bike relative to profile,Yes
2,Down_Payment,"Vehicle_Price, Loan_Amount",Price − Loan (≥ 0),Customer's own money in the deal; strongest si...,Yes
3,Approx_EMI,Loan_Amount,"Reducing-balance EMI, 24 months @ 22% p.a.",Monthly obligation,No: equals Loan_Amount × constant (ρ = 1)
4,EMI_to_Income,"Approx_EMI, Net_salary",EMI ÷ salary,FOIR-style repayment burden,No: depends on a question not asked (Net_salary)
5,Residual_Income,"Net_salary, Approx_EMI",Salary − EMI,Money left to live on,No: depends on a question not asked (Net_salary)
6,Down_Payment_to_Income,"Down_Payment, Net_salary",Down payment ÷ salary,Savings capacity,No: depends on a question not asked (Net_salary)
7,Age_Band,Age,18-25 / 26-35 / 36-45 / 46-60 / 60+,Non-linear age effect for LogReg,Yes
8,Is_First_Time_Borrower,PAST_LOANS_ACTIVE,1 if NO_PAST_LOANS,Thin credit file,Yes
9,High_LTV,LTV,1 if LTV ≥ 90%,Near-zero down payment red flag,Yes


## Deliverable 5: Minimal question set

In [25]:
QUESTION_TABLE

,Question,Features it feeds,AUC drop if removed (CV),AUC drop if removed (val),"Recall change if removed (val, @25% declined)",Keep?,Reason
3,Pincode,"pin_district, pin_zone, tier_imputed, tier_rank",0.0237,0.0222,-3.0%,Yes,Removal alone costs 0.0237 AUC (>= 0.005)
7,PAST_LOANS_ACTIVE,Is_First_Time_Borrower,0.0028,0.0047,-0.4%,Yes,"Cheap alone, but removing it on top of the oth..."
2,Age,"Age, Age_Band, age_above_policy",0.0010,0.0035,-0.2%,Yes,"Cheap alone, but removing it on top of the oth..."
1,Employment_Type,"Employment_Type, is_non_earner, salary_invalid",0.0008,0.0026,+0.0%,No,"Costs < 0.005 AUC on CV and validation, alone ..."
0,Qualifications,Qualifications,0.0005,0.0019,-0.1%,No,"Costs < 0.005 AUC on CV and validation, alone ..."
4,Gender,Gender,0.0002,0.0016,+0.6%,No,Removed by fairness policy (cost only +0.0002 ...
5,Resident_Type,Resident_Type,-0.0006,0.0013,+0.7%,No,"Costs < 0.005 AUC on CV and validation, alone ..."
6,Net_salary,"Down_Payment_to_Income, EMI_to_Income, Net_sal...",-0.0017,-0.0002,+0.2%,No,"Costs < 0.005 AUC on CV and validation, alone ..."


In [26]:
# Key numbers used in the written summary
summary = {
    "train rows": len(X_train), "val rows": len(X_val),
    "model": MODEL_KIND, "imbalance strategy": STRATEGY,
    "vehicle encoding": VEHICLE_CHOICE, "location encoding": ENC_CHOICE,
    "customer questions kept": KEEP_QUESTIONS, "removed": removed,
    "final val ROC-AUC": round(final_val["ROC-AUC"], 4), "final val PR-AUC": round(final_val["PR-AUC"], 4),
    "final val recall": round(final_val["Recall (DECLINED)"], 4),
    "final val precision": round(final_val["Precision (DECLINED)"], 4),
    "final % declined": round(final_val["% applicants declined"], 4),
    "threshold": round(FINAL_THRESHOLD, 4),
    "all-questions CV AUC": round(full["CV AUC"], 4),
    "final model columns": Xt_train.shape[1],
    "latency median ms": round(float(np.median(times)), 1),
}
json.dump(summary, open(f"{ART}/run_summary.json", "w"), indent=1, default=str)
summary

{'train rows': 86727,
 'val rows': 21682,
 'model': 'lgbm',
 'imbalance strategy': 'balanced',
 'vehicle encoding': 'A: Product + Make + Variant (one-hot)',
 'location encoding': 'pin_district one-hot (min_freq=30)',
 'customer questions kept': ['Age', 'Pincode', 'PAST_LOANS_ACTIVE'],
 'removed': ['Gender',
  'Net_salary',
  'Resident_Type',
  'Qualifications',
  'Employment_Type'],
 'final val ROC-AUC': 0.7858,
 'final val PR-AUC': 0.1832,
 'final val recall': 0.6044,
 'final val precision': 0.1362,
 'final % declined': np.float64(0.2333),
 'threshold': 0.5756,
 'all-questions CV AUC': 0.7813,
 'final model columns': 415,
 'latency median ms': 15.1}

## Deliverable 6: Summary for the report

### Feature engineering and selection
- **We engineered 13 credit-risk features** covering affordability (EMI-to-income, residual income), commitment (down payment, LTV ≥ 90%) and stability (first-time borrower, age band). EMI assumes a **24-month tenure at 22% p.a.** Every ratio is protected against zero or missing income.
- **The deal structure is the strongest risk signal.**
  - Decline rates run from **0.8% to 12.0%** across LTV quintiles (15x), and from **10.7% to 1.0%** across down-payment quintiles.
  - Salary on its own barely separates risk (1.1x). This is why features that combine fields matter.
- **We removed redundancy with evidence.**
  - EMI and EMI-to-income are exact rescalings of loan amount and loan-to-income (ρ = 1), so only one of each pair was kept.
  - Flags that merely regroup a category were dropped.
  - `Model_Description` added nothing over Make + Variant (CV AUC 0.778 either way), so the simpler one-hot set was kept.
  - The 3-digit pincode district, one-hot encoded, beat both target encoding and the full 13,000-level pincode.
- **Gender is excluded by policy.** Decline rates are 5.30% (female) vs 5.24% (male), chi-square p = 0.79, and removing it costs only 0.0002 AUC. It is kept in the data only for fairness monitoring.
- **The chatbot needs 3 customer questions instead of 8: Pincode, Age and Past loans.**
  - The other inputs come from the dealer quote, the catalogue or the pincode lookup.
  - Dropping Salary, Employment type, Qualification, Residence type and Gender costs **0.0049** validation AUC in total.
  - Pincode alone is worth 0.024 AUC.
  - If credit policy requires income for compliance, it can still be collected without feeding the score.

### Modelling approach
- **Leakage control:**
  - The data is split 80/20 (stratified, seed 42) before anything is learned.
  - Every learned step lives inside one pipeline fitted on train only: bike-price and tier lookups, caps, medians and encoders.
  - Target encoding is cross-fitted, and resampling is only ever applied inside CV training folds.
- **LightGBM with `class_weight="balanced"`** was chosen after comparing four imbalance strategies in 5-fold CV.
  - No adjustment and balanced weights tie (AUC 0.781). SMOTE (0.775) and undersampling (0.760) were worse.
  - Logistic regression was close behind (0.778), a useful, explainable benchmark.
- **The cut-off favours catching declines.** It maximises F2 on train out-of-fold scores. On validation the final 3-question model:
  - has ROC-AUC **0.786** and PR-AUC 0.183 (random = 0.053);
  - catches **60% of declines** while declining 23% of applicants;
  - has a precision of 13.6%, about 2.6x the base rate.
  The trade-off table in 6a lets the business pick a stricter or looser cut-off.
- **The model is production-ready.** It is one saved pipeline that takes a raw chatbot record. It handles blanks, unseen bike makes, invalid pincodes and over-age applicants without errors, and scores an applicant in about **20 ms**, far inside the 2-second target.